In [7]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("ARE_Silver_Transform")
    # keep ANSI enabled – we're showing we know how to handle it properly
    .config("spark.sql.ansi.enabled", "true")
    .getOrCreate()
)

raw_path    = "/Users/dev/PycharmProjects/jup/mnt/agri/bronze/parquet"
silver_path = "/Users/dev/PycharmProjects/jup/mnt/agri/silver/parquet"

# ---------------------------
# 1. Read bronze
# ---------------------------
farms_df   = spark.read.parquet(f"{raw_path}/farms")
sensors_df = spark.read.parquet(f"{raw_path}/sensor_readings")

# ---------------------------
# 2. Basic cleaning – farms
# ---------------------------
farms_clean = (
    farms_df
        .dropDuplicates(["farm_id"])
        .filter(F.col("farm_id").isNotNull())
)

# ---------------------------
# 3. Basic cleaning – sensors
#    (no timestamp parsing yet)
# ---------------------------
sensors_base = (
    sensors_df
        .dropDuplicates(["reading_id"])
        .filter(F.col("farm_id").isNotNull())
        .filter(F.col("reading_value").isNotNull())
)

# ---------------------------
# 4. Timestamp normalisation with DQ hooks
# ---------------------------
sensors_clean = (
    sensors_base
        # keep the original raw string
        .withColumn("reading_ts_raw", F.col("reading_ts").cast("string"))
        # normalise spaces, trim
        .withColumn(
            "reading_ts_clean_str",
            F.regexp_replace(F.trim(F.col("reading_ts_raw")), r"\s+", " ")
        )
        # parse using multiple patterns, tolerant of bad values
        .withColumn(
            "reading_ts_parsed",
            F.expr(
                """
                coalesce(
                    try_to_timestamp(reading_ts_clean_str, 'dd/MM/yyyy HH:mm:ss'),
                    try_to_timestamp(reading_ts_clean_str, 'dd/MM/yyyy HH:mm'),
                    try_to_timestamp(reading_ts_clean_str, 'yyyy-MM-dd HH:mm:ss')
                )
                """
            )
        )
        # DQ flag: we had a non-null string but could not parse it
        .withColumn(
            "reading_ts_parse_error",
            (F.col("reading_ts_clean_str").isNotNull()) &
            (F.col("reading_ts_parsed").isNull())
        )
        # final canonical timestamp column used downstream
        .withColumn("reading_ts", F.col("reading_ts_parsed"))
        .drop("reading_ts_parsed")  # keep raw/clean + final
)

# ---------------------------
# 5. Simple DQ metrics (for your README / demo)
# ---------------------------
total_count  = sensors_base.count()
error_count  = sensors_clean.filter(F.col("reading_ts_parse_error") == True).count()

print(f"[DQ] Sensors total: {total_count}, timestamp parse errors: {error_count}")

# ---------------------------
# 6. Split by sensor type
# ---------------------------
soil_moisture_df = sensors_clean.filter(F.col("sensor_type") == "SoilMoisture")
temperature_df   = sensors_clean.filter(F.col("sensor_type") == "Temperature")

print("[INFO] Soil moisture schema")
soil_moisture_df.printSchema()

print("[INFO] Temperature schema")
temperature_df.printSchema()

# ---------------------------
# 7. Persist Silver layer
# ---------------------------
(
    farms_clean
        .write
        .mode("overwrite")
        .parquet(f"{silver_path}/farms")
)

(
    sensors_clean
        .write
        .mode("overwrite")
        .parquet(f"{silver_path}/sensor_readings")
)

(
    soil_moisture_df
        .write
        .mode("overwrite")
        .parquet(f"{silver_path}/soil_moisture")
)

(
    temperature_df
        .write
        .mode("overwrite")
        .parquet(f"{silver_path}/temperature")
)

spark.stop()


[DQ] Sensors total: 2000, timestamp parse errors: 1
[INFO] Soil moisture schema
root
 |-- reading_id: string (nullable = true)
 |-- farm_id: string (nullable = true)
 |-- sensor_type: string (nullable = true)
 |-- reading_value: double (nullable = true)
 |-- reading_ts: timestamp (nullable = true)
 |-- reading_ts_raw: string (nullable = true)
 |-- reading_ts_clean_str: string (nullable = true)
 |-- reading_ts_parse_error: boolean (nullable = false)

[INFO] Temperature schema
root
 |-- reading_id: string (nullable = true)
 |-- farm_id: string (nullable = true)
 |-- sensor_type: string (nullable = true)
 |-- reading_value: double (nullable = true)
 |-- reading_ts: timestamp (nullable = true)
 |-- reading_ts_raw: string (nullable = true)
 |-- reading_ts_clean_str: string (nullable = true)
 |-- reading_ts_parse_error: boolean (nullable = false)

